# Testing
here we will test each functions and each code so we understand what each function is doing and how are they doing it.

In [52]:
# Train Ensemble

# Imports
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np

from pathlib import Path
from typing import Any
from scipy.stats import spearmanr


In [ ]:
class SurrogateNN(nn.Module):
    def __init__(self, input_sim=4, output_dim=6):
        super(SurrogateNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_sim, 64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)
    
def train_model(X_train, Y_train, seed, epochs=100):
    """
    This function trains a single NN.
    """
    g = torch.Generator()
    g.manual_seed(seed)
    np.random.seed(seed)
    
    # creating the model from SurrogateNN class that we created at the top
    model = SurrogateNN()
    # selecting the criteria in which the loss will be measured
    criterion = nn.MSELoss()
    # selecting the optmizer for gradient
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Loading the dataset(scaled)
    dataset = TensorDataset(X_train, Y_train)
    loader  = DataLoader(dataset, batch_size=32, shuffle=True, generator=g)
    
    # setting the module in training mode
    model.train()
    # starting the training loop for epochs iterations
    for epoch in range(epochs):
        if epoch % 100 == 0:
            print(f"[x]    Model training -> Epoch: {epoch}")
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss =criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    return model


def train_ensemble(X_train,
                   Y_train,
                   X_test:Any|None=None,
                   y_test:Any|None=None,
                   bounds:Any|None=None,
                   epochs:Any|None=None,
                   seed:Any|None=None,
                   num_members:int|None=None, 
                   storage_path:Path|None=None, 
                   save:bool|None = True,
                   evaluate:bool|None=True):
    ensemble = []
    for i in range(num_members):
        print(f"[x]    Training model number: {i}")
        m = train_model(X_train, Y_train, seed+i, epochs=epochs)
        ensemble.append(m)
    print(f"[x]    Training complete. Saving model")
    try:
        if save:
            torch.save(ensemble, storage_path)
            print(f"[x]    Models saved at {storage_path}")
    except Exception as e:
        print(f"[x]    Model saving failed. ERROR: {e}")    


In [54]:
def main():
    root = Path.cwd().parent
    
    # loading scaling bounds
    bounds_path = root / "data" / "processed" / "scaling_bounds.json"
    with open(bounds_path, 'r', encoding='utf-8') as file:
        bounds = json.load(file)
    
    print(bounds)
    print(type(bounds))
    
    # load the train and test set
    train_file  =   root / "data" / "processed" / "train_1200.csv"
    test_file   =   root / "data" / "processed" / "test_set.csv"
    
    # model file
    model_file  =   root / "models" / "ensemble_1200_my_own_test.pth"
    
    df_train = pd.read_csv(train_file)
    df_test  = pd.read_csv(test_file)
    
    # data is already scaled to [-1, 1] form process.py
    X_train =   torch.tensor(df_train.iloc[:, :4].values, dtype=torch.float32)
    Y_train =   torch.tensor(df_train.iloc[:, 4:].values, dtype=torch.float32)
    X_test  =   torch.tensor(df_test.iloc[:, :4].values, dtype=torch.float32)
    Y_test  =   torch.tensor(df_test.iloc[:, 4:].values, dtype=torch.float32)
    
    
    # training the ensemble
    ensemble = train_ensemble(X_train, 
                              Y_train, 
                              num_members=20,
                              save=True,
                              evaluate=False,
                              epochs=1000,
                              seed=42,
                              storage_path=model_file)

if __name__ == "__main__":
    main()

{'min': {'beam_width': 0.2013688722879548, 'beam_length': 10.0017117591943, 'fillet_radius': 0.1009545147605954, 'beam_height': 5.000267675912441, 'max_stress_von_mises': 0.0939508571288682, 'max_deformation': 0.0021283109832839, 'mode_1_freq': 23.02788820149861, 'mode_2_freq': 31.202172347275955, 'mode_3_freq': 49.60955596653007, 'mode_4_freq': 51.09410939366549}, 'max': {'beam_width': 1.999947632811813, 'beam_length': 39.97573020615898, 'fillet_radius': 1.999229677628755, 'beam_height': 19.9931295840198, 'max_stress_von_mises': 13.842807670326136, 'max_deformation': 4.947364481720527, 'mode_1_freq': 1139.2662734493913, 'mode_2_freq': 1399.9096245875346, 'mode_3_freq': 2018.3103491664792, 'mode_4_freq': 2248.947758690331}, 'columns': ['beam_width', 'beam_length', 'fillet_radius', 'beam_height', 'max_stress_von_mises', 'max_deformation', 'mode_1_freq', 'mode_2_freq', 'mode_3_freq', 'mode_4_freq']}
<class 'dict'>
[x]    Training model number: 0
[x]    Model training -> Epoch: 0
[x]    M

KeyboardInterrupt: 

In [ ]:
root = Path.cwd().parent
model_path = root / "models" / "ensemble_1200_my_own_test.pth"
test_file   =   root / "data" / "processed" / "test_set.csv"

# bounds
bounds_file = root / "data" / "processed" / "scaling_bounds.json"
with open(bounds_file, encoding='utf-8') as file:
    bounds = json.load(file)

ensemble = torch.load(model_path, weights_only=False)
df_test  = pd.read_csv(test_file)

X_test  =   torch.tensor(df_test.iloc[:, :4].values, dtype=torch.float32)
Y_test  =   torch.tensor(df_test.iloc[:, 4:].values, dtype=torch.float32)

# predicting output forward pass
for m in ensemble: m.eval()
with torch.no_grad():
    preds_scaled = torch.stack([m(X_test) for m in ensemble]).numpy()

mean_pred_scaled = np.mean(preds_scaled, axis=0)
std_pred_scaled  = np.std(preds_scaled, axis=0)



# Reverse scaling for the "Physical" error calculation
# Formula: x = ((x_scaled + 1) * (max - min) / 2) + min
cols = bounds["columns"]
# We only care about the output columns (index 4 onwards)
output_cols = cols[4:] 
mins = np.array([bounds["min"][c] for c in output_cols])
maxs = np.array([bounds["max"][c] for c in output_cols])
ranges = maxs - mins

print(f"Max values of output columns: \n{maxs}")
print(f"Min values of output columns: \n{mins}")



# descale the target and mean prediction
y_test_raw = ((Y_test + 1)*ranges/2) + mins
mean_pred_raw = ((mean_pred_scaled + 1) * ranges / 2) + mins




def evaluate_ensemble(ensemble, 
                      X_test, 
                      y_test_scaled, 
                      bounds):
    # sets every model to evaluation mode. This diables dropout or batch
    # normalization(though not used in this architecture) ensuring deterministic
    # output
    for m in ensemble: m.eval()
    
    # torch.no_grad() stops pytorch from creating gradients to save memory. It runs 
    # the test data through all 20 models, stack the results into a single tensor,
    # and converts it to a NumPy array. Shape(20 models, N samples, 6 Outputs)
    with torch.no_grad():
        preds_scaled = torch.stack([m(X_test) for m in ensemble]).numpy()
    
    # calculates the average prediction (the ensembles final answer) and 
    # the standard deviation (the disagreement) across the 20 models(models number i's 
    # outputs are at preds_scaled[i,:,:])
    mean_pred_scaled = np.mean(preds_scaled, axis=0)
    std_pred_scaled = np.std(preds_scaled, axis=0)
    
    # Physical descaling
    # Reverse scaling for the  "Physical" error calculation
    # Formula: x = ((x_scaled + 1) *(max - min)/2) + min
    cols =bounds["columns"]
    
    # now we only care about he output columns
    output_cols = cols[4:]
    print(output_cols)

Max values of output columns: 
[  21.88385398   17.54440448 1139.26627345 1399.90962459 2018.31034917
 2248.94775869]
Min values of output columns: 
[9.39508571e-02 2.12831098e-03 1.30588168e+01 1.81276602e+01
 2.60482508e+01 2.67873715e+01]


C:\Users\Administrator\AppData\Local\Temp\ipykernel_10512\2321060483.py:41: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  y_test_raw = ((Y_test + 1)*ranges/2) + mins


In [51]:
def clean_csv(csv_filePath: Path, save_as_filename: str) -> pd.DataFrame:
    """Cleans the raw CSV by removing empty strings and dropping NaNs."""
    if not csv_filePath.exists():
        raise FileNotFoundError(f"Dataset not found at: {csv_filePath}")
    
    df = pd.read_csv(csv_filePath)
    df = df.replace(r'^\s*$', np.nan, regex=True).dropna(axis=0, how='any')
    
    # dropping the columns that has larger than 5mm in max_deformation
    if "max_deformation" in df.columns:
        before = len(df)
        df = df[df["max_deformation"] <= 5].reset_index(drop=True)
        print(f"[clean_csv] Dropped {before - len(df)} rows with max_deformation > 5")
        
    
    target = csv_filePath.parent / save_as_filename
    if not target.suffix:
        target = target.with_suffix('.csv')
    
    df.to_csv(target, index=False, encoding='utf-8')
    print(f"saved cleaned file to {target}")
    return df

raw_input = root / "data" / "raw" / "inputs.csv"

clean_csv(raw_input, "cleaned.csv")

[clean_csv] Dropped 77 rows with max_deformation > 5
saved cleaned file to c:\Users\Administrator\Desktop\code\comp_design_fw\data\raw\cleaned.csv


,Design_ID,beam_width,beam_length,fillet_radius,beam_height,status,max_stress_von_mises,max_deformation,mode_1_freq,mode_2_freq,mode_3_freq,mode_4_freq
0,1,0.942572,17.329495,1.127268,10.175530,completed,0.666178,0.052899,227.483723,287.436273,464.665779,486.103184
1,2,1.647421,20.629430,0.940045,18.658316,completed,0.434306,0.025537,331.303163,498.509751,711.392299,881.443742
2,3,1.583590,23.389746,1.220573,13.279694,completed,0.536506,0.039795,265.921316,395.980793,558.722458,706.985707
3,4,1.639956,10.478294,0.231410,13.402796,completed,0.327069,0.004511,779.091110,994.940927,1400.455476,1608.870384
4,5,1.429232,13.025015,0.442878,10.110812,completed,0.364933,0.010230,517.799491,677.515184,943.784996,1079.283049
...,...,...,...,...,...,...,...,...,...,...,...,...
1815,1996,0.307480,14.508981,0.785536,14.918192,completed,4.032192,0.629091,65.141572,68.297250,127.970494,149.241351
1816,1997,1.990446,18.470591,1.282554,9.146828,completed,0.301297,0.012059,482.846117,723.119134,960.875870,1020.856908
1817,1998,1.486505,14.343615,1.300118,6.381075,completed,0.247356,0.010333,516.627346,661.458585,812.648180,1037.152568
1818,1999,1.940073,10.046132,0.357296,19.627927,completed,0.203728,0.002844,983.667436,1311.674550,1949.662140,2084.381531
